# Uso de MultiLCA

Este cuaderno demuestra:

+ como restaurar un proyecto con bases de datos incluídas
+ como calcular multiples LCIA para una misma unidad funcional

In [ ]:
import bw2data as bd
import bw2io as bi
import bw2calc as bc

In [ ]:
print(f"bw2data version -> {bd.__version__}")
print(bi.__version__)
print(bc.__version__)


In [ ]:
bd.projects


In [ ]:
project_names = list(bd.projects)

In [ ]:
for p_name in project_names:
    if "mob" in str(p_name):
        print(p_name)

    

## Restaurar el proyecto

In [ ]:
bi.restore_project_directory("brightway2-project-mobility-backup.03-February-2025-11-38PM.tar.gz")

In [ ]:
bd.projects.current

In [ ]:
bd.projects.set_current("mobility")


In [ ]:
bd.projects.current

In [ ]:
bd.databases

## Multiples LCIA

### Seleccionar multiples métodos de impacto

In [ ]:
recipe_mid_h = []
for nombre_metodo in bd.methods:
    if "ReC" in nombre_metodo[0] and "midpoint" in nombre_metodo[0] and "H" in nombre_metodo[0]:
        print(nombre_metodo)
        recipe_mid_h.append(nombre_metodo)

### Buscar una actividad en la base de datos

In [ ]:
actividades_bike = []
for actividad in bd.Database("ecoinvent-3.10-cutoff"):
    if "bicycle" in actividad["name"]:
        print(actividad)
        actividades_bike.append(actividad)
actividades_bike = sorted(actividades_bike, key=lambda actividad: actividad["name"])

In [ ]:
for a in actividades_bike:
    print(f"{a['name']} id -> {a['code']}")

In [ ]:
bike_a = bd.Database("ecoinvent-3.10-cutoff").get("4bc540cbbd84cc38be7413bfcce79927")
bike_a

In [ ]:
for e in bike_a.exchanges():
    print(e)

### Crear una configuración de cálculo para múltiples LCIA

In [ ]:
bd.calculation_setups?

In [ ]:
cs_name = "cs_recipe_bike"

In [ ]:
bd.calculation_setups[cs_name] = {
    'inv': [{bike_a.key:1}],
    'ia':recipe_mid_h
}

In [ ]:
bc.MultiLCA?

#### Ejecutar el cálculo

In [ ]:
mlca = bc.MultiLCA(cs_name)

In [ ]:
for i in mlca.results:
    print(i)

In [ ]:
for i in mlca.methods:
    print(i)

## Ejercicios

1. hacer un cálculo para un solo método, pero muchas unidades funcionales (seleccionar una bicicleta eléctrica)
2. hacer un cálculo para muchas unidades funcionales y muchos métodos

## Análisis de contribuciones con bwa - guía austera

In [ ]:
import bw2analyzer as bwa

In [ ]:
bwa.print_recursive_supply_chain?

In [ ]:
bwa.print_recursive_supply_chain(bike_a)

In [ ]:
bwa.print_recursive_calculation?

In [ ]:
bwa.print_recursive_calculation(bike_a, recipe_mid_h[0])

### Ejercicios

1. Hacer un análisis de contribución para el método IPCC para la bike_a